<a href="https://colab.research.google.com/github/brenonsx066-ship-it/Dr.-comunnity/blob/main/banco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
from datetime import datetime

class DatabaseManager:
    def __init__(self, db_name="dr_community.db"):
        self.conexao = sqlite3.connect(db_name)
        self.cursor = self.conexao.cursor()
        self.cursor.execute("PRAGMA foreign_keys = ON")
        self.criar_tabelas()

    def criar_tabelas(self):
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS pacientes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL UNIQUE,
            senha TEXT NOT NULL,
            idade INTEGER NOT NULL,
            queixa TEXT NOT NULL,
            niveldedor INTEGER NOT NULL,
            status TEXT NOT NULL
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS medicos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            crm TEXT NOT NULL UNIQUE,
            especialidade TEXT NOT NULL,
            senha TEXT NOT NULL
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS agendamentos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            paciente_id INTEGER NOT NULL,
            medico_id INTEGER,
            medico TEXT NOT NULL,
            especialidade TEXT NOT NULL,
            data TEXT NOT NULL,
            horario TEXT NOT NULL,
            tipo TEXT NOT NULL,
            status TEXT NOT NULL DEFAULT 'Pendente',
            FOREIGN KEY (paciente_id) REFERENCES pacientes(id),
            FOREIGN KEY (medico_id) REFERENCES medicos(id)
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS exames (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            paciente_id INTEGER NOT NULL,
            nome TEXT NOT NULL,
            data TEXT NOT NULL,
            resultado TEXT NOT NULL,
            observacao TEXT DEFAULT '',
            FOREIGN KEY (paciente_id) REFERENCES pacientes(id)
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS mensagens (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            paciente_id INTEGER NOT NULL,
            medico_id INTEGER,
            remetente TEXT NOT NULL,
            mensagem TEXT NOT NULL,
            data_envio TEXT NOT NULL,
            lida INTEGER NOT NULL DEFAULT 0,
            FOREIGN KEY (paciente_id) REFERENCES pacientes(id),
            FOREIGN KEY (medico_id) REFERENCES medicos(id)
        )
        """)
        self.conexao.commit()

    def fechar(self):
        self.conexao.close()


class AtendimentoModel:
    def __init__(self, db_manager):
        self.db = db_manager

    def cadastrar_paciente(self, nome, senha, idade, queixa, niveldedor, status):
        self.db.cursor.execute("""
        INSERT INTO pacientes
        (nome, senha, idade, queixa, niveldedor, status)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (nome, senha, idade, queixa, niveldedor, status))
        self.db.conexao.commit()

    def listar_cadastros(self):
        self.db.cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        ORDER BY id
        """)
        return self.db.cursor.fetchall()

    def buscar_paciente_por_login(self, nome, senha):
        self.db.cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        WHERE LOWER(nome) = LOWER(?) AND senha = ?
        """, (nome, senha))
        return self.db.cursor.fetchone()

    def buscar_paciente_por_id(self, paciente_id):
        self.db.cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        WHERE id = ?
        """, (paciente_id,))
        return self.db.cursor.fetchone()

    def atualizar_dados_saude(self, queixa, dor, status, paciente_id):
        self.db.cursor.execute("""
        UPDATE pacientes
        SET queixa = ?, niveldedor = ?, status = ?
        WHERE id = ?
        """, (queixa, dor, status, paciente_id))
        self.db.conexao.commit()

    def listar_medicos(self):
        self.db.cursor.execute("""
        SELECT id, nome, especialidade
        FROM medicos
        ORDER BY nome
        """)
        return self.db.cursor.fetchall()

    def buscar_medico_por_id(self, medico_id):
        self.db.cursor.execute("""
        SELECT id, nome, especialidade
        FROM medicos
        WHERE id = ?
        """, (medico_id,))
        return self.db.cursor.fetchone()

    def buscar_medicos_por_termo(self, termo):
        self.db.cursor.execute("""
        SELECT id, nome, especialidade
        FROM medicos
        WHERE nome LIKE ? OR especialidade LIKE ?
        ORDER BY nome
        """, (f"%{termo}%", f"%{termo}%"))
        return self.db.cursor.fetchall()

    def buscar_medicos_detalhado(self, nome, especialidade):
        self.db.cursor.execute("""
        SELECT id, nome, especialidade
        FROM medicos
        WHERE nome LIKE ?
        AND especialidade LIKE ?
        ORDER BY nome
        """, (f"%{nome}%", f"%{especialidade}%"))
        return self.db.cursor.fetchall()

    def verificar_conflito_agendamento(self, medico_id, data, horario):
        self.db.cursor.execute("""
        SELECT id FROM agendamentos
        WHERE medico_id = ?
        AND data = ?
        AND horario = ?
        AND status = 'Confirmado'
        """, (medico_id, data, horario))
        return self.db.cursor.fetchone()

    def criar_agendamento(self, paciente_id, medico_id, nome_medico, especialidade, data, horario, tipo):
        self.db.cursor.execute("""
        INSERT INTO agendamentos
        (paciente_id, medico_id, medico, especialidade,
         data, horario, tipo, status)
        VALUES (?, ?, ?, ?, ?, ?, ?, 'Pendente')
        """, (
            paciente_id, medico_id, nome_medico,
            especialidade, data, horario, tipo
        ))
        self.db.conexao.commit()

    def listar_agendamentos_paciente(self, paciente_id):
        self.db.cursor.execute("""
        SELECT id, medico, especialidade, data,
               horario, tipo, status
        FROM agendamentos
        WHERE paciente_id = ?
        ORDER BY id DESC
        """, (paciente_id,))
        return self.db.cursor.fetchall()

    def buscar_agendamento_por_id(self, consulta_id, paciente_id):
        self.db.cursor.execute("""
        SELECT status
        FROM agendamentos
        WHERE id = ? AND paciente_id = ?
        """, (consulta_id, paciente_id))
        return self.db.cursor.fetchone()

    def cancelar_agendamento(self, consulta_id, paciente_id):
        self.db.cursor.execute("""
        UPDATE agendamentos
        SET status = 'Cancelado'
        WHERE id = ? AND paciente_id = ?
        """, (consulta_id, paciente_id))
        self.db.conexao.commit()

    def listar_exames_paciente(self, paciente_id):
        self.db.cursor.execute("""
        SELECT id, nome, data, resultado, observacao
        FROM exames
        WHERE paciente_id = ?
        ORDER BY id DESC
        """, (paciente_id,))
        return self.db.cursor.fetchall()

    def verificar_medico_existe(self, medico_id):
        self.db.cursor.execute(
            "SELECT id FROM medicos WHERE id = ?",
            (medico_id,)
        )
        return self.db.cursor.fetchone()

    def salvar_mensagem(self, paciente_id, medico_id, mensagem, data_envio):
        self.db.cursor.execute("""
        INSERT INTO mensagens
        (paciente_id, medico_id, remetente, mensagem, data_envio)
        VALUES (?, ?, 'Paciente', ?, ?)
        """, (
            paciente_id, medico_id, mensagem, data_envio
        ))
        self.db.conexao.commit()

    def listar_mensagens_paciente(self, paciente_id):
        self.db.cursor.execute("""
        SELECT mensagens.remetente,
               medicos.nome,
               mensagens.mensagem,
               mensagens.data_envio
        FROM mensagens
        LEFT JOIN medicos ON mensagens.medico_id = medicos.id
        WHERE mensagens.paciente_id = ?
        ORDER BY mensagens.id
        """, (paciente_id,))
        return self.db.cursor.fetchall()

    def listar_notificacoes_paciente(self, paciente_id):
        self.db.cursor.execute("""
        SELECT medico, data, horario, status
        FROM agendamentos
        WHERE paciente_id = ?
        ORDER BY id DESC
        """, (paciente_id,))
        return self.db.cursor.fetchall()